# Lab: Probability & Uncertainty in AI

## Case Study: Autonomous Deep-Space Rescue AI System

### Scenario

You are designing an AI system for an **Autonomous Deep-Space Rescue Mission**.

A rescue ship must decide whether to launch based on uncertain environmental and system conditions.

Due to incomplete information, the AI must reason probabilistically.

---

## Random Variables

Each variable represents a set of possible worlds (ω):

- Weather (W) = {calm, storm}
- Solar Radiation (S) = {low, high}
- Engine Status (E) = {working, faulty}
- Navigation System (N) = {accurate, drift}
- Distress Signal (D) = {true, false}
- Rescue Success (R) = {success, fail}

---

## AI Goals

The system must:

- Compute probabilities of rescue success
- Update beliefs using Bayes’ Rule
- Detect independence between variables
- Compute joint probabilities
- Perform marginalization
- Validate probability distributions
- Handle uncertainty in decision-making

---

## Input File Formats

All input files follow a simple **comma-separated format (CSV without headers)**.

Each file represents a probability distribution or joint probability table.

---

### 1. Distribution Files

Files:
- weather.txt  
- solar.txt  
- engine.txt  
- navigation.txt  
- distress.txt  
- rescue.txt  

**Format:**

state,probability


**Notes:**
- Each line represents one possible world (ω)
- Probabilities must be between 0 and 1
- The sum of all probabilities in a file must equal **1**

---

### 2. Joint Probability File

File:
- joint.txt  

**Format:**

weather,solar,engine,rescue,probability


---

### 3. Important Rules

- Joint probabilities must sum to **1**
- Each row represents a **complete possible world**
- Not all combinations need to be present (missing ones assumed 0)
- Order of variables must be consistent


---


---

## Learning Objectives

By completing this lab, you will:

- Model uncertainty using probability distributions
- Implement conditional probability and Bayes’ Rule
- Work with joint distributions
- Apply marginalization and conditioning
- Test independence of events
- Validate probability axioms in real systems

## Step 1: Build the Knowledge Base

Implement a complete probability knowledge base using OOP.

In [26]:
class RescueKB:
    def __init__(self):
        self.distributions = {
            "weather": {},
            "solar": {},
            "engine": {},
            "navigation": {},
            "distress": {},
            "rescue": {}
        }

        # Joint probabilities:
        # (W, S, E, R) → probability
        self.joint = {}


    def add_probability(self, var, state, prob):
        self.distributions[var][state] = prob

    def add_joint(self, w, s, e, r, prob):
        self.joint[(w, s, e, r)] = prob


    def P(self, var, state):
        return self.distributions[var].get(state, 0.0)

    def get_distribution(self, var):
        return self.distributions[var]


    def validate_distribution(self, var):
        """
        Check if probabilities sum to 1
        """
        dist = self.distributions[var]
        total = sum(dist.values())
        return abs(total - 1.0) < 1e-6

    def validate_all(self):
        """
        Validate all distributions
        """
        for var in self.distributions:
            if not self.validate_distribution(var):
                raise ValueError(f"Invalid distribution for {var}")

## Step 2: File Readers with Validation

In [27]:
def read_distribution(file, var, kb):
    """
    Format:
    state,prob
    """
    with open(file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                state, prob = line.split(',')
                kb.add_probability(var, state, float(prob))


def read_joint(file, kb):
    """
    Format:
    weather,solar,engine,rescue,prob
    """
    with open(file, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                w, s, e, r, prob = line.split(',')
                kb.add_joint(w, s, e, r, float(prob))

## Step 3: Probability Engine

Implement all probability rules.

In [28]:
class ProbabilityEngine:
    def __init__(self, kb):
        self.kb = kb


    def negation(self, p):
        """
        P(~A) = 1 - P(A)
        """
        return 1 - p


    def inclusion_exclusion(self, p_a, p_b, p_ab):
        """
        P(A ∪ B) = P(A) + P(B) - P(A ∩ B)
        """
        return p_a + p_b - p_ab


    def joint_probability(self, conditions):
        """
        conditions: dict
        Example: {"weather": "storm", "engine": "faulty"}
        """
        total = 0.0
        for key, prob in self.kb.joint.items():
            w, s, e, r = key
            match = True
            for var, state in conditions.items():
                if var == "weather" and w != state:
                    match = False
                    break
                elif var == "solar" and s != state:
                    match = False
                    break
                elif var == "engine" and e != state:
                    match = False
                    break
                elif var == "rescue" and r != state:
                    match = False
                    break
            if match:
                total += prob
        return total


    def marginal(self, query_var, query_state):
        """
        Sum over joint distribution
        """
        return self.joint_probability({query_var: query_state})


    def conditional(self, query, evidence):
        """
        P(A | B)
        """
        combined = {**query, **evidence}
        p_combined = self.joint_probability(combined)
        p_evidence = self.joint_probability(evidence)
        if p_evidence == 0:
            return 0.0
        return p_combined / p_evidence


    def bayes(self, p_a, p_b_given_a, p_b):
        if p_b == 0:
            return 0.0
        return (p_b_given_a * p_a) / p_b


    def is_independent(self, p_a, p_b, p_ab):
        return abs(p_ab - p_a * p_b) < 1e-6

## Step 4: Evaluations

In [29]:
class Evaluator:
    def __init__(self, kb):
        self.kb = kb
        self.engine = ProbabilityEngine(kb)

    def probability_of_success(self):
        return self.engine.marginal("rescue", "success")

    def success_given_storm(self):
        return self.engine.conditional({"rescue": "success"}, {"weather": "storm"})

    def bayes_engine_given_failure(self):
        """
        P(E=faulty | R=fail)
        """
        p_faulty = self.kb.P("engine", "faulty")
        p_fail_given_faulty = self.engine.conditional({"rescue": "fail"}, {"engine": "faulty"})
        p_fail = self.engine.marginal("rescue", "fail")
        return self.engine.bayes(p_faulty, p_fail_given_faulty, p_fail)

    def check_weather_engine_independence(self):
        p_storm = self.engine.marginal("weather", "storm")
        p_faulty = self.engine.marginal("engine", "faulty")
        p_storm_and_faulty = self.engine.joint_probability({"weather": "storm", "engine": "faulty"})
        return self.engine.is_independent(p_storm, p_faulty, p_storm_and_faulty)

    def marginal_solar(self):
        return self.engine.marginal("solar", "low")

    def inclusion_exclusion_test(self):
        """
        P(storm ∪ faulty engine)
        """
        p_storm = self.engine.marginal("weather", "storm")
        p_faulty = self.engine.marginal("engine", "faulty")
        p_both = self.engine.joint_probability({"weather": "storm", "engine": "faulty"})
        return self.engine.inclusion_exclusion(p_storm, p_faulty, p_both)

## Step 5: Output Writer

Do NOT modify.

In [30]:
def write_output(results, filename="output.txt"):
    output = "Deep Space Rescue AI Results\n"
    output += "----------------------------\n"

    for r in results:
        output += r + "\n"

    print(output)

    with open(filename, "w", encoding='utf-8') as f:
        f.write(output)

## Step 6: Exercises

In [31]:
# Q1: Compute P(Distress = true | Storm)

def distress_given_storm(kb):
    # Assuming independence
    return kb.P("distress", "true")


# Q2: Check if Solar and Weather are independent

def solar_weather_independent(kb):
    engine = ProbabilityEngine(kb)
    p_low = engine.marginal("solar", "low")
    p_calm = engine.marginal("weather", "calm")
    p_low_and_calm = engine.joint_probability({"solar": "low", "weather": "calm"})
    return engine.is_independent(p_calm, p_low, p_low_and_calm)


# Q3:
# Modify input so engine failure increases.
# What happens to rescue success?

# TODO: Answer
# Increasing engine failure probability would decrease the overall rescue success probability,
# as faulty engines are associated with lower success rates in the joint distribution.


# Q4: Find MOST dangerous scenario
# (max probability of failure)

def most_dangerous_scenario(kb):
    max_prob = 0
    scenario = None
    for key, prob in kb.joint.items():
        w, s, e, r = key
        if r == "fail" and prob > max_prob:
            max_prob = prob
            scenario = (w, s, e, r)
    return scenario


# Q5:
# Compute P(R) using conditioning:
# P(R) = Σ P(R|W)P(W)

def rescue_using_conditioning(kb):
    engine = ProbabilityEngine(kb)
    total = 0
    for w_state in kb.get_distribution("weather"):
        p_w = kb.P("weather", w_state)
        p_success_given_w = engine.conditional({"rescue": "success"}, {"weather": w_state})
        total += p_success_given_w * p_w
    return total


# Q6:
# Normalize a distribution manually

def normalize_distribution(dist):
    total = sum(dist.values())
    if total == 0:
        return dist
    return {k: v / total for k, v in dist.items()}


# Q7:
# Detect invalid distributions

def find_invalid_distributions(kb):
    invalid = []
    for var in kb.distributions:
        if not kb.validate_distribution(var):
            invalid.append(var)
    return invalid


# Q8:
# Compute full joint sum (should be 1)

def verify_joint_sum(kb):
    total = sum(kb.joint.values())
    return abs(total - 1.0) < 1e-6

## Step 7: Main Function

Do NOT modify.

In [32]:
def evaluate(kb):
    evaluator = Evaluator(kb)

    results = []

    try:
        results.append("P(Success): " + str(evaluator.probability_of_success()))
    except:
        results.append("P(Success): NOT IMPLEMENTED")

    try:
        results.append("P(Success | Storm): " + str(evaluator.success_given_storm()))
    except:
        results.append("P(Success | Storm): NOT IMPLEMENTED")

    try:
        results.append("Bayes Engine|Failure: " + str(evaluator.bayes_engine_given_failure()))
    except:
        results.append("Bayes Engine|Failure: NOT IMPLEMENTED")

    try:
        results.append("Weather ⟂ Engine: " + str(evaluator.check_weather_engine_independence()))
    except:
        results.append("Weather ⟂ Engine: NOT IMPLEMENTED")

    return results


def main():
    kb = RescueKB()

    read_distribution("weather.txt", "weather", kb)
    read_distribution("solar.txt", "solar", kb)
    read_distribution("engine.txt", "engine", kb)
    read_distribution("navigation.txt", "navigation", kb)
    read_distribution("distress.txt", "distress", kb)
    read_distribution("rescue.txt", "rescue", kb)

    read_joint("joint.txt", kb)

    kb.validate_all()

    results = evaluate(kb)

    # Exercises
    try:
        results.append("Distress | Storm: " + str(distress_given_storm(kb)))
    except:
        results.append("Distress | Storm: NOT IMPLEMENTED")

    try:
        results.append("Solar ⟂ Weather: " + str(solar_weather_independent(kb)))
    except:
        results.append("Solar ⟂ Weather: NOT IMPLEMENTED")

    try:
        results.append("Most Dangerous Scenario: " + str(most_dangerous_scenario(kb)))
    except:
        results.append("Most Dangerous Scenario: NOT IMPLEMENTED")

    try:
        results.append("Rescue via Conditioning: " + str(rescue_using_conditioning(kb)))
    except:
        results.append("Rescue via Conditioning: NOT IMPLEMENTED")

    try:
        results.append("Invalid Distributions: " + str(find_invalid_distributions(kb)))
    except:
        results.append("Invalid Distributions: NOT IMPLEMENTED")

    try:
        results.append("Joint Sum Check: " + str(verify_joint_sum(kb)))
    except:
        results.append("Joint Sum Check: NOT IMPLEMENTED")

    write_output(results)


if __name__ == "__main__":
    main()

Deep Space Rescue AI Results
----------------------------
P(Success): 0.375
P(Success | Storm): 0.34375
Bayes Engine|Failure: 0.19856845993996763
Weather ⟂ Engine: False
Distress | Storm: 0.4
Solar ⟂ Weather: False
Most Dangerous Scenario: ('extreme', 'high', 'faulty', 'fail')
Rescue via Conditioning: 0.5038467333654773
Invalid Distributions: []
Joint Sum Check: False

